# TP1 — Neural Networks & Model Comparison (Heart Disease)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/DFGSM2-IA-S2/blob/main/session3/tp1.ipynb)

## Objective

In this final TP, we bring together everything from sessions 1 and 2:
1. Apply the full **data preparation pipeline** (session 2) to a new dataset
2. Train and compare **three models**: Logistic Regression, Random Forest, and **Neural Network (MLP)**
3. Learn how neural networks work and how to tune them

### New Dataset: Heart Disease

We use the **Heart Disease (Statlog)** dataset — 270 patients with 13 clinical features. The task is to predict the **presence or absence of heart disease**.

Some features are truly numeric (age, blood pressure), while others are integers encoding categories (chest pain type, thal). This is common in real medical datasets.

### Methodology Recap

Same pipeline as always: **Explore → Visualize → Prepare → Train → Evaluate → Compare**

> **Link to previous sessions:** Session 1 taught evaluation metrics, session 2 taught data preparation. This session applies both and adds a new model.

---
## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, f1_score)

import warnings
warnings.filterwarnings('ignore')

---
## 2. Load & Discover the Data

The **Heart Disease (Statlog)** dataset contains clinical data from 270 patients.

**13 features:**
- `age` — Age in years
- `sex` — Sex (1 = male, 0 = female)
- `chest` — Chest pain type (1-4)
- `resting_blood_pressure` — Resting blood pressure (mm Hg)
- `serum_cholestoral` — Serum cholesterol (mg/dl)
- `fasting_blood_sugar` — Fasting blood sugar > 120 mg/dl (1 = yes, 0 = no)
- `resting_electrocardiographic_results` — Resting ECG results (0-2)
- `maximum_heart_rate_achieved` — Maximum heart rate achieved
- `exercise_induced_angina` — Exercise-induced angina (1 = yes, 0 = no)
- `oldpeak` — ST depression induced by exercise
- `slope` — Slope of peak exercise ST segment (1-3)
- `number_of_major_vessels` — Number of major vessels colored by fluoroscopy (0-3)
- `thal` — Thalassemia type (3, 6, or 7)

**Target:** `present` or `absent` — presence of heart disease.

### Exercise 1.1 — Load the dataset

Load the Heart Disease dataset and explore its shape, features, and first rows.

*Hint:* `fetch_openml("heart-statlog", version=1, as_frame=True)`. The target column is called `class`.

In [ ]:
# YOUR CODE HERE


### Exercise 1.2 — Summary statistics and data types

Display summary statistics and check data types. Note which features are truly continuous vs integer-encoded categories.

*Hint:* `df.info()`, `df.describe()`

**Question:** Which features represent categories even though they are stored as numbers?

In [ ]:
# YOUR CODE HERE


*YOUR ANSWER HERE*

### Exercise 1.3 — Check class balance

Count how many patients have heart disease vs. don't.

**Why:** Same reason as session 1 — check if accuracy is a reliable metric.

*Hint:* `y.value_counts(normalize=True)`

In [ ]:
# YOUR CODE HERE


---
## 3. Data Visualization

Same methodology as sessions 1-2: understand the data visually before modeling.

### Exercise 2.1 — Feature distributions by class

Plot the distribution of `age` and `maximum_heart_rate_achieved`, colored by class.

**Why:** See if these clinical features differ between patients with and without heart disease.

*Hint:* `sns.histplot(data=df, x='age', hue='target', kde=True)`

In [ ]:
# YOUR CODE HERE


### Exercise 2.2 — Scatter plot

Create a scatter plot of `age` vs `maximum_heart_rate_achieved`, colored by class.

**Why:** See if these two features together help separate the classes.

*Hint:* `sns.scatterplot(x='age', y='maximum_heart_rate_achieved', hue='target', data=df, alpha=0.6)`

In [ ]:
# YOUR CODE HERE


### Exercise 2.3 — Correlation heatmap

Compute and display a correlation heatmap for all features.

**Why:** Identify relationships between features and with the target.

*Hint:* `sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')`

**Question:** Which features seem most correlated with heart disease?

In [ ]:
# YOUR CODE HERE


*YOUR ANSWER HERE*

---
## 4. Data Preparation

Apply what you learned in session 2. This dataset has no missing values and all features are already numeric, so we only need to **scale** the features.

**Note:** Some features (sex, chest, fasting_blood_sugar, etc.) are integer-encoded categories. For tree-based models this doesn't matter, but for logistic regression and neural networks, scaling is still important.

### Exercise 3.1 — Split and scale

Split into train/test (80/20, `random_state=42`) and scale with `StandardScaler`.

*Hint:* Same pattern as sessions 1 and 2.

In [ ]:
# YOUR CODE HERE


---
## 5. Baseline: Logistic Regression

Always start with a baseline. Logistic regression is our reference model from session 1.

### Exercise 4.1 — Train logistic regression

Train a logistic regression and report accuracy, confusion matrix, and classification report.

*Hint:* `LogisticRegression(max_iter=5000, random_state=42)`

In [ ]:
# YOUR CODE HERE


---
## 6. Random Forest

Apply what you learned in session 2 — train a random forest and compare with the baseline.

### Exercise 5.1 — Train a Random Forest

Train a `RandomForestClassifier(n_estimators=100, random_state=42)` and compare with logistic regression.

*Hint:* Same as session 2.

In [ ]:
# YOUR CODE HERE


---
## 7. Neural Network (MLPClassifier)

### What is a Neural Network?

A **neural network** is inspired by the brain: layers of interconnected "neurons" that transform the input step by step.

### Architecture

```
Input Layer        Hidden Layer(s)       Output Layer
(features)         (learned repr.)       (prediction)

  x₁ ──┐         ┌── h₁ ──┐
        ├─────────┤         ├──────── ŷ (probability)
  x₂ ──┤         ├── h₂ ──┤
        ├─────────┤         ├
  x₃ ──┤         ├── h₃ ──┘
        ├─────────┤
  ...   ┘         └── ...
```

Each connection has a **weight** (like θ in logistic regression). The network **learns** these weights during training.

### Key Differences from Logistic Regression

| | Logistic Regression | Neural Network (MLP) |
|---|---|---|
| **Layers** | 1 (input → output) | Multiple (input → hidden → output) |
| **Decision boundary** | Linear | Non-linear (any shape!) |
| **Parameters** | Few (n_features + 1) | Many (depends on architecture) |
| **Training** | Fast, convex optimization | Slower, non-convex (can get stuck) |
| **Scaling required** | Yes | **Critical** (even more important) |

### MLPClassifier in sklearn

`MLPClassifier` = **Multi-Layer Perceptron** Classifier

Key hyperparameters:
- `hidden_layer_sizes` — e.g., `(100,)` = 1 hidden layer with 100 neurons, `(64, 32)` = 2 layers
- `activation` — `'relu'` (default), `'tanh'`, `'logistic'`
- `learning_rate_init` — initial learning rate (default 0.001)
- `max_iter` — maximum training iterations

> **Medical analogy:** If logistic regression is like a single doctor making a diagnosis based on test results, a neural network is like a team of specialists, each focusing on different patterns, combining their insights for a final diagnosis.

### Exercise 6.1 — Train a basic neural network

Train an `MLPClassifier` with default parameters and compare with previous models.

*Hint:* `MLPClassifier(max_iter=1000, random_state=42)`

In [ ]:
# YOUR CODE HERE


### Exercise 6.2 — Visualize the loss curve

Plot the training loss curve to see how the network learned.

**Why:** The loss curve shows whether the model converged (loss decreasing and stabilizing) or needs more iterations.

*Hint:* `plt.plot(mlp_model.loss_curve_)`, `plt.xlabel('Iteration')`, `plt.ylabel('Loss')`

In [ ]:
# YOUR CODE HERE


---
## 8. Tuning the Neural Network

### Experimenting with Architecture

The architecture of a neural network (number and size of hidden layers) greatly affects its performance:

| Architecture | Description | When to use |
|-------------|-------------|-------------|
| `(50,)` | 1 layer, 50 neurons | Simple problems |
| `(100,)` | 1 layer, 100 neurons | Default, good starting point |
| `(64, 32)` | 2 layers (64 → 32) | More complex patterns |
| `(128, 64, 32)` | 3 layers | Complex problems, more data needed |

**Rule of thumb:** Start simple, add complexity only if needed. More layers ≠ better performance (especially with small datasets).

### Exercise 7.1 — Experiment with architectures

Try different `hidden_layer_sizes` and compare results. Plot the loss curves together.

*Hint:*
```python
architectures = [(50,), (100,), (64, 32), (128, 64, 32)]
for arch in architectures:
    model = MLPClassifier(hidden_layer_sizes=arch, max_iter=1000, random_state=42)
    model.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_scaled))
    print(f"{str(arch):<20} Accuracy: {acc:.4f}")
    plt.plot(model.loss_curve_, label=str(arch))
```

In [ ]:
# YOUR CODE HERE


### Exercise 7.2 — Grid search for best MLP

Use `GridSearchCV` to find the best combination of `hidden_layer_sizes`, `activation`, and `learning_rate_init`.

*Hint:*
```python
param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (64, 32)],
    'activation': ['relu', 'tanh'],
    'learning_rate_init': [0.001, 0.01]
}
grid_search_mlp = GridSearchCV(MLPClassifier(max_iter=1000, random_state=42),
                                param_grid, cv=5, scoring='accuracy', n_jobs=-1)
```

In [ ]:
# YOUR CODE HERE


---
## 9. Final Model Comparison

Now let's compare all three models systematically.

### Exercise 8.1 — Cross-validation comparison

Run 5-fold cross-validation for all three models and create a comparison table.

*Hint:*
```python
models = {
    'Logistic Regression': LogisticRegression(max_iter=5000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Neural Network (MLP)': grid_search_mlp.best_estimator_
}
for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
    print(f"{name:<25} {scores.mean():.4f} ± {scores.std():.4f}")
```

In [ ]:
# YOUR CODE HERE


### Exercise 8.2 — Confusion matrices comparison

Display confusion matrices for all three models side by side.

*Hint:* `fig, axes = plt.subplots(1, 3, figsize=(18, 5))`

In [ ]:
# YOUR CODE HERE


---
## 10. Summary — Three Sessions of ML

### The Complete ML Pipeline

```
Session 1                    Session 2                     Session 3
─────────                    ─────────                     ─────────
Clean data                   Noisy data                    New dataset
Logistic Regression          + Data Preparation            + Neural Network
Evaluation metrics           + Random Forest               + Model Comparison
                             + Hyperparameter Tuning
```

### Models Comparison

| | Logistic Regression | Random Forest | Neural Network (MLP) |
|---|---|---|---|
| **Type** | Linear | Ensemble of trees | Layers of neurons |
| **Boundary** | Linear | Non-linear | Non-linear |
| **Scaling** | Important | Not needed | Critical |
| **Interpretability** | High (coefficients) | Medium (feature importances) | Low (black box) |
| **Training speed** | Fast | Medium | Slow |
| **When to use** | Baseline, linearly separable data | Tabular data, robust default | Complex patterns, large data |

### The ML Methodology (applies to ALL models)

1. **Explore** → Understand your data (shape, types, distributions)
2. **Visualize** → See patterns, outliers, relationships
3. **Prepare** → Handle missing values, outliers, encode categories, scale
4. **Split** → Train/test for honest evaluation
5. **Train** → Start with baseline, then try more complex models
6. **Evaluate** → Use appropriate metrics (accuracy, precision, recall, F1)
7. **Tune** → Optimize hyperparameters with cross-validation
8. **Compare** → Select the best model for your specific problem

### Key Takeaway for Medical Applications

> In healthcare, **the choice of metric matters more than the choice of model**. A model with 95% accuracy but low recall for disease detection is dangerous. Always consider the **cost of errors** — a missed diagnosis (false negative) is typically much worse than a false alarm (false positive).